In [ ]:
!pip install -q transformers datasets accelerate scikit-learn pandas

In [ ]:
import pandas as pd
train_df = pd.read_csv('trainV2.csv')
test_df = pd.read_csv('testV2.csv')
train_df['Class'] = train_df['Class'].str.strip()
mapping = {
    'Non-Abusive': 1,
    'Abusive': 0,
    'abusive': 0
}
train_df['label'] = train_df['Class'].map(mapping)
train_df = train_df.dropna(subset=['label'])
train_df['label'] = train_df['label'].astype(int)
print("Label Distribution:")
print(train_df['label'].value_counts())
print("\nFirst 5 rows of Train:")
print(train_df[['Text', 'label']].head())

Label Distribution:
label
1    1883
0    1769
Name: count, dtype: int64

First 5 rows of Train:
                                                Text  label
0  நான் கூட உன்னை வெகுளியான பொண்ணு&#39;னு நெனச்சி...      1
1  உன் போட்டோவை டாய்லெட்டுக்கு மாற்றினார்கள் அசிங...      0
2  கண்டா வரச்சொல்லுங்க கார்த்திய திவ்யாவோட சேர்த்...      1
3  ஒன்னோட சைசுக்கு நீயே ஒரு நாளக்கி 5கிலோ ஆய் போவ...      0
4  ரெண்டும் மிக பெரிய வெடிகுண்டு இவங்கள எதுக்கு ஷ...      1


In [ ]:
from sklearn.model_selection import train_test_split
from datasets import Dataset, DatasetDict
from transformers import AutoTokenizer
model_name = "xlm-roberta-base"
tokenizer = AutoTokenizer.from_pretrained(model_name)
train_sub_df, val_df = train_test_split(train_df, test_size=0.2, random_state=42, stratify=train_df['label'])
dataset = DatasetDict({
    'train': Dataset.from_pandas(train_sub_df[['Text', 'label']].reset_index(drop=True)),
    'validation': Dataset.from_pandas(val_df[['Text', 'label']].reset_index(drop=True)),
    'test': Dataset.from_pandas(test_df[['Text']].reset_index(drop=True))
})
def tokenize_function(examples):
    return tokenizer(examples["Text"], padding="max_length", truncation=True, max_length=128)
tokenized_datasets = dataset.map(tokenize_function, batched=True)
tokenized_datasets = tokenized_datasets.remove_columns(["Text"])
tokenized_datasets.set_format("torch")
print("Tokenization complete.")

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:104: UserWarning: 
Error while fetching `HF_TOKEN` secret value from your vault: 'Requesting secret HF_TOKEN timed out. Secrets can only be fetched when running from the Colab UI.'.
You are not authenticated with the Hugging Face Hub in this notebook.
If the error persists, please let us know by opening an issue on GitHub (https://github.com/huggingface/huggingface_hub/issues/new).
  warnings.warn(


config.json:   0%|          | 0.00/615 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/25.0 [00:00<?, ?B/s]

sentencepiece.bpe.model:   0%|          | 0.00/5.07M [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/9.10M [00:00<?, ?B/s]

Map:   0%|          | 0/2921 [00:00<?, ? examples/s]

Map:   0%|          | 0/731 [00:00<?, ? examples/s]

Map:   0%|          | 0/913 [00:00<?, ? examples/s]

Tokenization complete.


In [ ]:
from transformers import AutoModelForSequenceClassification, TrainingArguments, Trainer
import numpy as np
from sklearn.metrics import accuracy_score, precision_recall_fscore_support
model = AutoModelForSequenceClassification.from_pretrained(model_name, num_labels=2)
def compute_metrics(eval_pred):
    logits, labels = eval_pred
    predictions = np.argmax(logits, axis=-1)
    precision, recall, f1, _ = precision_recall_fscore_support(labels, predictions, average='binary')
    acc = accuracy_score(labels, predictions)
    return {
        'accuracy': acc,
        'f1': f1,
        'precision': precision,
        'recall': recall
    }
training_args = TrainingArguments(
    output_dir="./results",
    eval_strategy="epoch",
    save_strategy="epoch",
    learning_rate=2e-5,
    per_device_train_batch_size=16,
    per_device_eval_batch_size=16,
    num_train_epochs=3,
    weight_decay=0.01,
    load_best_model_at_end=True,
    metric_for_best_model="accuracy",
    fp16=True,
    logging_steps=10,
    report_to="none"
)
trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=tokenized_datasets["train"],
    eval_dataset=tokenized_datasets["validation"],
    compute_metrics=compute_metrics,
)

print("Model and Trainer are initialized and ready for training.")

Loading weights:   0%|          | 0/197 [00:00<?, ?it/s]

XLMRobertaForSequenceClassification LOAD REPORT from: xlm-roberta-base
Key                         | Status     | 
----------------------------+------------+-
lm_head.bias                | UNEXPECTED | 
lm_head.layer_norm.weight   | UNEXPECTED | 
lm_head.dense.weight        | UNEXPECTED | 
roberta.pooler.dense.weight | UNEXPECTED | 
lm_head.layer_norm.bias     | UNEXPECTED | 
lm_head.dense.bias          | UNEXPECTED | 
roberta.pooler.dense.bias   | UNEXPECTED | 
classifier.dense.bias       | MISSING    | 
classifier.dense.weight     | MISSING    | 
classifier.out_proj.weight  | MISSING    | 
classifier.out_proj.bias    | MISSING    | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING	:those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


Model and Trainer are initialized and ready for training.


In [ ]:
trainer.train()
print("Training complete! Evaluating on validation set...")
trainer.evaluate()

Epoch,Training Loss,Validation Loss,Accuracy,F1,Precision,Recall
1,0.617528,0.611584,0.700410,0.629442,0.869159,0.493369
2,0.517480,0.507391,0.766074,0.740516,0.865248,0.647215
3,0.445067,0.503491,0.787962,0.790823,0.804945,0.777188


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

There were missing keys in the checkpoint model loaded: ['roberta.embeddings.LayerNorm.weight', 'roberta.embeddings.LayerNorm.bias', 'roberta.encoder.layer.0.attention.output.LayerNorm.weight', 'roberta.encoder.layer.0.attention.output.LayerNorm.bias', 'roberta.encoder.layer.0.output.LayerNorm.weight', 'roberta.encoder.layer.0.output.LayerNorm.bias', 'roberta.encoder.layer.1.attention.output.LayerNorm.weight', 'roberta.encoder.layer.1.attention.output.LayerNorm.bias', 'roberta.encoder.layer.1.output.LayerNorm.weight', 'roberta.encoder.layer.1.output.LayerNorm.bias', 'roberta.encoder.layer.2.attention.output.LayerNorm.weight', 'roberta.encoder.layer.2.attention.output.LayerNorm.bias', 'roberta.encoder.layer.2.output.LayerNorm.weight', 'roberta.encoder.layer.2.output.LayerNorm.bias', 'roberta.encoder.layer.3.attention.output.LayerNorm.weight', 'roberta.encoder.layer.3.attention.output.LayerNorm.bias', 'roberta.encoder.layer.3.output.LayerNorm.weight', 'roberta.encoder.layer.3.output.Laye

Training complete! Evaluating on validation set...


{'eval_loss': 0.5034905076026917,
 'eval_accuracy': 0.7879616963064295,
 'eval_f1': 0.7908232118758435,
 'eval_precision': 0.804945054945055,
 'eval_recall': 0.7771883289124668,
 'eval_runtime': 1.3213,
 'eval_samples_per_second': 553.228,
 'eval_steps_per_second': 34.813,
 'epoch': 3.0}

In [1]:
import torch

# 1. Get predictions from the test set
print("Predicting classes for test dataset...")
test_predictions = trainer.predict(tokenized_datasets["test"])

# 2. Convert logits to class indices (0 or 1)
preds = np.argmax(test_predictions.predictions, axis=-1)

# 3. Create a mapping to convert 1 -> Non-Abusive and 0 -> Abusive
reverse_mapping = {1: 'Non-Abusive', 0: 'Abusive'}
predicted_labels = [reverse_mapping[p] for p in preds]

# 4. Create the final DataFrame
# Assuming test_df still contains the original 'Text' column
output_df = pd.DataFrame({
    'Text': test_df['Text'],
    'Predicted_Class': predicted_labels
})

# 5. Save to CSV
output_filename = "test_predictions.csv"
output_df.to_csv(output_filename, index=False)

print(f"Predictions saved successfully to {output_filename}!")

# Display the first few rows of the result
output_df.head()

Predicting classes for test dataset...


NameError: name 'trainer' is not defined